# Phase 1: Understanding the Data (Customer Churn)

**Goal of this notebook:** understand the dataset well enough to explain it to a panel in 30 seconds, before any cleaning, EDA or modeling.

**Order of work (same order to follow in the hackathon):**
1. Read the problem statement: goal, target column, churn definition
2. Load the data and check shape and structure
3. Work out the grain (what one row means)
4. Check data quality (missing, duplicates, impossible values)
5. Check target balance
6. Classify columns (numeric, categorical, ID, target)
7. Write hypotheses
8. Write the spoken summary

**Business goal:** find which customers are likely to churn and why, so the retention team can act early.

## Step 1: Load the data
**What we do:** read the CSV into a pandas DataFrame.
**Why:** everything starts from the raw data. We do not change anything yet.

In [ ]:
import pandas as pd

df = pd.read_csv("customer_churn_dataset-testing-master.csv")
df.head()

**What we learned:** each row is one customer with demographic, subscription, usage, support, payment and spend columns. The last column, `Churn`, is the target.

## Step 2: Size and structure
**What we do:** check `shape` and `info()`.
**Why:** shape tells us the size, and info() shows data types and non-null counts. A number stored as text would show up here.

In [ ]:
print("Rows, columns:", df.shape)
df.info()

**What we learned:** 64,374 rows and 12 columns. `Gender`, `Subscription Type` and `Contract Length` are text (categorical). Every other column is a number. No column has a wrong type.

## Step 3: Grain (what does one row mean?)
**What we do:** check whether `CustomerID` is unique.
**Why:** if it is unique, one row equals one customer. If we got the grain wrong, every count and percentage later would be wrong.

In [ ]:
print("Unique CustomerIDs:", df["CustomerID"].nunique())
print("Total rows        :", len(df))
print("One row per customer?", df["CustomerID"].nunique() == len(df))

**What we learned:** the grain is one row per customer. There is only one table, so there are no joins to do.

## Step 4: Data quality
**What we do:** count missing values and duplicate rows, then look at `describe()`.
**Why:** missing values and duplicates break models. `describe()` shows min and max, so impossible values (negative age, absurd spend) are visible. We only list problems here. Fixing is Phase 2.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.describe().T

**What we learned:**
- No missing values and no duplicate rows.
- Min and max values are sensible: age 18 to 65, support calls 0 to 10, payment delay 0 to 30, tenure 1 to 60, total spend 100 to 1000.
- Mean and median are close for every numeric column, so there is no heavy skew.
- The 25%, 50% and 75% points are evenly spaced for most columns. That looks like simulated data, which we confirm with histograms in Phase 3.

In [ ]:
# Explicit check for impossible values
print("Age outside 18-65       :", ((df["Age"] < 18) | (df["Age"] > 65)).sum())
print("Negative support calls  :", (df["Support Calls"] < 0).sum())
print("Negative payment delay  :", (df["Payment Delay"] < 0).sum())
print("Total spend <= 0        :", (df["Total Spend"] <= 0).sum())
print("Tenure <= 0             :", (df["Tenure"] <= 0).sum())

**What we learned:** zero impossible values. The data quality is good, so Phase 2 (cleaning) will be short.

## Step 5: Target balance
**What we do:** check the percentage of churned vs retained customers.
**Why:** if one class is rare (say 5%), accuracy becomes misleading and we need class weights or SMOTE. If the classes are balanced, accuracy is fair.

In [ ]:
print(df["Churn"].value_counts())
print()
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

**What we learned:** about 47.4% churned and 52.6% stayed. The classes are almost balanced, so no SMOTE or class weights are needed. We will still report precision, recall and F1 along with accuracy.

## Step 6: Classify the columns
**What we do:** split columns into groups.
**Why:** numeric and categorical columns need different graphs, tests and encoding later.

In [ ]:
id_col = ["CustomerID"]
target = ["Churn"]
categorical = ["Gender", "Subscription Type", "Contract Length"]
numeric = ["Age", "Tenure", "Usage Frequency", "Support Calls",
           "Payment Delay", "Total Spend", "Last Interaction"]

print("Categorical unique values:")
for c in categorical:
    print(" ", c, df[c].unique().tolist())
print()
print("Numeric columns:", numeric)

| Group | Columns | Used for |
|---|---|---|
| ID | CustomerID | Drop before modeling, it is only an identifier |
| Target | Churn | What we predict (1 = left, 0 = stayed) |
| Categorical | Gender, Subscription Type, Contract Length | Needs encoding for the model, chi-square test against Churn |
| Numeric | Age, Tenure, Usage Frequency, Support Calls, Payment Delay, Total Spend, Last Interaction | Histograms, boxplots, t-test or Mann-Whitney against Churn |

Assumed units (confirm with the panel): Tenure in months, Payment Delay and Last Interaction in days.

## Step 7: Hypotheses (to test in Phase 3)
Written before analysis so the work looks planned:
1. Customers with more support calls churn more.
2. Customers with longer payment delay churn more.
3. Monthly contract customers churn more than annual ones.
4. Customers with longer tenure churn less.
5. Customers with lower usage frequency churn more.

## Step 8: 30-second spoken summary (for the panel)
"The dataset has about 64,000 customers, one row per customer, with 11 features covering demographics, subscription, usage, support, payment and spend. The target is Churn, and about 47% of customers churned, so the classes are nearly balanced. There are no missing values, duplicates or impossible values. I will drop CustomerID since it is only an identifier. My hypotheses are that support calls, payment delay and contract type drive churn, and I will test them statistically next."

## Likely panel questions on this phase
- **What does one row represent?** One customer, because CustomerID is unique.
- **Why drop CustomerID?** It is unique per row and carries no pattern, so the model could fit noise.
- **Is the data imbalanced?** No, about 47/53, so accuracy is meaningful, but we also report precision, recall and F1.
- **Which columns are categorical and which numeric?** Gender, Subscription Type and Contract Length are categorical. The rest, except the ID and target, are numeric.
- **How did you check data quality?** `isnull().sum()`, `duplicated().sum()`, and min/max from `describe()` plus explicit checks for impossible values.

**Next:** Phase 2, data cleaning and preparation.